# Multi-Component Flash Separation (BTX)

Simulating an isothermal flash drum for a ternary **benzene–toluene–p-xylene** (BTX) mixture.
The `MultiComponentFlash` block uses Raoult's law with Antoine correlations to compute K-values
and solves the Rachford-Rice equation via Brent's method.

This example is inspired by [MiniSim's SimpleFlash example](https://github.com/Nukleon84/MiniSim/blob/master/doc/SimpleFlash.ipynb),
adapted to PathSim's dynamic simulation framework.

**Feed conditions:**
- 10 mol/s total flow
- 50% benzene, 10% toluene, 40% p-xylene (molar)
- 1 atm pressure
- Temperature sweep: 340 K → 420 K

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('../pathsim_docs.mplstyle')

from pathsim import Simulation, Connection
from pathsim.blocks import Source, Scope

from pathsim_chem.process import MultiComponentFlash

## Flash Drum Setup

The `MultiComponentFlash` block defaults to BTX Antoine parameters (ln form, Pa, K):

| Component | A | B | C |
|-----------|-------|---------|--------|
| Benzene | 20.7936 | 2788.51 | -52.36 |
| Toluene | 20.9064 | 3096.52 | -53.67 |
| p-Xylene | 20.9891 | 3346.65 | -57.84 |

We feed the drum with constant composition and pressure while ramping temperature
to observe the transition from all-liquid through two-phase to all-vapor.

In [ ]:
# Create the flash drum (3 components, BTX defaults)
flash = MultiComponentFlash(N_comp=3, holdup=100.0)

# Feed sources
src_f = Source(lambda t: 10.0)          # 10 mol/s
src_z1 = Source(lambda t: 0.5)          # 50% benzene
src_z2 = Source(lambda t: 0.1)          # 10% toluene (p-xylene = 40% inferred)
src_t = Source(lambda t: 340.0 + t)     # ramp 340 -> 420 K
src_p = Source(lambda t: 101325.0)      # 1 atm

# Record all outputs: V_rate, L_rate, y_benzene, y_toluene, x_benzene, x_toluene
sco = Scope(labels=["V_rate", "L_rate", "y_benz", "y_tol", "x_benz", "x_tol"])

sim = Simulation(
    [src_f, src_z1, src_z2, src_t, src_p, flash, sco],
    [
        Connection(src_f, flash),            # F
        Connection(src_z1, flash[1]),        # z_1 (benzene)
        Connection(src_z2, flash[2]),        # z_2 (toluene)
        Connection(src_t, flash[3]),         # T
        Connection(src_p, flash[4]),         # P
        Connection(flash, sco),              # V_rate
        Connection(flash[1], sco[1]),        # L_rate
        Connection(flash[2], sco[2]),        # y_benzene
        Connection(flash[3], sco[3]),        # y_toluene
        Connection(flash[4], sco[4]),        # x_benzene
        Connection(flash[5], sco[5]),        # x_toluene
    ],
    dt=0.5,
    log=True,
)

sim.run(80)

## Results: Flow Rates

As temperature increases, the vapor fraction grows. Below the bubble point the drum produces
only liquid; above the dew point it produces only vapor.

In [ ]:
time, [V_rate, L_rate, y_benz, y_tol, x_benz, x_tol] = sco.read()
T = 340.0 + time  # temperature axis

# Infer p-xylene fractions
y_xyl = 1.0 - y_benz - y_tol
x_xyl = 1.0 - x_benz - x_tol

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Flow rates
ax = axes[0]
ax.plot(T, V_rate, label="Vapor")
ax.plot(T, L_rate, label="Liquid")
ax.set_xlabel("Temperature [K]")
ax.set_ylabel("Flow rate [mol/s]")
ax.set_title("Flash Drum Flow Rates")
ax.legend()
ax.grid(True, alpha=0.3)

# Vapor compositions
ax = axes[1]
ax.plot(T, y_benz, label="Benzene")
ax.plot(T, y_tol, label="Toluene")
ax.plot(T, y_xyl, label="p-Xylene")
ax.set_xlabel("Temperature [K]")
ax.set_ylabel("Vapor mole fraction")
ax.set_title("Vapor Composition")
ax.legend()
ax.grid(True, alpha=0.3)

# Liquid compositions
ax = axes[2]
ax.plot(T, x_benz, label="Benzene")
ax.plot(T, x_tol, label="Toluene")
ax.plot(T, x_xyl, label="p-Xylene")
ax.set_xlabel("Temperature [K]")
ax.set_ylabel("Liquid mole fraction")
ax.set_title("Liquid Composition")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Results: VLE Diagram

Plot the vapor vs liquid composition for each component across the temperature sweep.
The diagonal represents equal vapor and liquid composition — deviation from it shows
the separation achieved by the flash.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, xi, yi, name in zip(axes,
                             [x_benz, x_tol, x_xyl],
                             [y_benz, y_tol, y_xyl],
                             ["Benzene", "Toluene", "p-Xylene"]):
    ax.plot(xi, yi, ".", markersize=3)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_xlabel(f"$x$ ({name})")
    ax.set_ylabel(f"$y$ ({name})")
    ax.set_title(f"{name} (x, y)-Diagram")
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Fixed-Temperature Flash at 380 K

For a direct comparison with MiniSim's result (which solves at steady state),
we run a fixed-temperature flash and let the holdup reach equilibrium.

In [ ]:
flash2 = MultiComponentFlash(N_comp=3, holdup=100.0)

src_f2 = Source(lambda t: 10.0)
src_z1b = Source(lambda t: 0.5)
src_z2b = Source(lambda t: 0.1)
src_t2 = Source(lambda t: 380.0)       # fixed at 380 K (~107 °C)
src_p2 = Source(lambda t: 101325.0)

sco2 = Scope(labels=["V_rate", "L_rate", "y_benz", "y_tol", "x_benz", "x_tol"])

sim2 = Simulation(
    [src_f2, src_z1b, src_z2b, src_t2, src_p2, flash2, sco2],
    [
        Connection(src_f2, flash2),
        Connection(src_z1b, flash2[1]),
        Connection(src_z2b, flash2[2]),
        Connection(src_t2, flash2[3]),
        Connection(src_p2, flash2[4]),
        Connection(flash2, sco2),
        Connection(flash2[1], sco2[1]),
        Connection(flash2[2], sco2[2]),
        Connection(flash2[3], sco2[3]),
        Connection(flash2[4], sco2[4]),
        Connection(flash2[5], sco2[5]),
    ],
    dt=0.5,
    log=True,
)

sim2.run(100)  # let it reach steady state

_, [V_ss, L_ss, y_benz_ss, y_tol_ss, x_benz_ss, x_tol_ss] = sco2.read()

print("BTX Flash at 380 K, 1 atm")
print("=" * 40)
print(f"{'':20s} {'Vapor':>10s} {'Liquid':>10s}")
print(f"{'-'*40}")
print(f"{'Flow rate [mol/s]':20s} {V_ss[-1]:10.3f} {L_ss[-1]:10.3f}")
print(f"{'Benzene':20s} {y_benz_ss[-1]:10.4f} {x_benz_ss[-1]:10.4f}")
print(f"{'Toluene':20s} {y_tol_ss[-1]:10.4f} {x_tol_ss[-1]:10.4f}")
print(f"{'p-Xylene':20s} {1-y_benz_ss[-1]-y_tol_ss[-1]:10.4f} {1-x_benz_ss[-1]-x_tol_ss[-1]:10.4f}")

The lighter component (benzene) is enriched in the vapor phase while the heavier
component (p-xylene) concentrates in the liquid — exactly the separation behaviour
expected from VLE. The dynamic formulation reaches the same steady state that
an equation-oriented solver (like MiniSim) finds directly.